# 100 - Recurrent Networks: `nn.RNN`

**Section:** RNNs | **Prereqs:** `ANNs/NNClasses.ipynb` | **Next:** `RNNs/Predicting_Alternating_Sequences.ipynb`

Every model so far treats its input as a fixed-size block. An RNN processes a
*sequence* one step at a time, carrying a **hidden state** forward:

    h_t = tanh(W_ih @ x_t + W_hh @ h_{t-1} + b)

`h_t` is the network's memory. The same weights are applied at every time step -
the temporal analogue of a CNN's weight sharing across space - so the model works
on sequences of any length.

**The shape convention that catches everyone.** PyTorch's default is
`(seq_len, batch, features)`, with the sequence axis *first*. Every other layer in
PyTorch puts batch first. Pass `batch_first=True` to get `(batch, seq_len,
features)` instead, and be consistent - mismatched conventions here produce
silently wrong results rather than errors, because the shapes often still fit.

**`nn.RNN` returns two things:** `output` (the hidden state at *every* time step,
shape `(seq_len, batch, hidden)`) and `h_n` (only the final one, shape
`(num_layers, batch, hidden)`). Use `output[-1]` for one prediction per sequence,
all of `output` for one per step.

**In practice, use LSTM or GRU.** A plain RNN multiplies by `W_hh` at every step,
so gradients vanish or explode over long sequences and it cannot learn
dependencies more than ~10 steps back. LSTM and GRU add gates that let information
pass unchanged, and they are drop-in replacements (`nn.LSTM` returns
`(h_n, c_n)` rather than just `h_n`).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np

In [ ]:
# nn.RNN's arguments: features per time step, hidden state size, how many
# stacked recurrent layers, and the non-linearity ('tanh' or 'relu').
# tanh is the default because it keeps the hidden state bounded, which matters
# when the same weights are applied over and over.
rnn = nn.RNN(input_size=9, hidden_size=16 , num_layers=1, nonlinearity='tanh',bias=True)
print(rnn)

RNN(9, 16)


In [ ]:
# Two return values: the output at every time step, and the final hidden state.
rnn(torch.randn(1,9))[0].shape ,rnn(torch.randn(1,9))[1].shape

(torch.Size([1, 16]), torch.Size([1, 16]))

In [ ]:
??nn.RNN

In [ ]:
# THE SHAPE CONVENTION: (seq_len, batch, features) - sequence axis FIRST.
# Every other PyTorch layer puts batch first; pass batch_first=True to switch.
# The hidden state is (num_layers, batch, hidden_size) - it has no sequence axis,
# because there is only one hidden state at a time.
seq_length = 5
batchsize = 2

X = torch.randn(seq_length,batchsize, 9) # batchsize,seq_length, input_length

hidden = torch.zeros(1,batchsize,16) # num_layer,batchsize,hidden_size

y,h = rnn(X,hidden)

In [ ]:
# The three shapes side by side. Getting these right is most of the work when
# writing recurrent code.
print(f'Input Shape: {list(X.shape)}')
print(f'Hidden Shape: {list(h.shape)}')
print(f'Output Shape: {list(y.shape)}')

Input Shape: [5, 2, 9]
Hidden Shape: [1, 2, 16]
Output Shape: [5, 2, 16]


In [ ]:
# Passing the hidden state explicitly vs letting PyTorch default it to zeros -
# identical results, so the explicit zeros are optional at the start of a
# sequence. You DO need to pass it when continuing a sequence across calls.
y,h1 = rnn(X,hidden)

y,h2 = rnn(X) # This automatically intialaizes hidden state to be zeros.

print(h1)
print('\n\n')
print(h2)
print('\n')


print(f"Are the weights same : {torch.equal(h1,h2)}")

tensor([[[-0.6108,  0.8248, -0.5500,  0.5908, -0.0635,  0.0839,  0.1325,
          -0.7241,  0.1945, -0.8424,  0.0093, -0.7923, -0.1896, -0.5581,
          -0.3360,  0.1898],
         [ 0.2423,  0.7203, -0.5469,  0.7187, -0.8546,  0.2797, -0.4001,
          -0.4501, -0.1565, -0.8735, -0.1598, -0.8250, -0.0108, -0.5140,
          -0.9373,  0.2612]]], grad_fn=<StackBackward0>)



tensor([[[-0.6108,  0.8248, -0.5500,  0.5908, -0.0635,  0.0839,  0.1325,
          -0.7241,  0.1945, -0.8424,  0.0093, -0.7923, -0.1896, -0.5581,
          -0.3360,  0.1898],
         [ 0.2423,  0.7203, -0.5469,  0.7187, -0.8546,  0.2797, -0.4001,
          -0.4501, -0.1565, -0.8735, -0.1598, -0.8250, -0.0108, -0.5140,
          -0.9373,  0.2612]]], grad_fn=<StackBackward0>)


Are the weights same : True


In [ ]:
# The weight shapes:
#   weight_ih_l0 (hidden, input)  - input to hidden,
#   weight_hh_l0 (hidden, hidden) - hidden to hidden, the recurrent connection.
# It is the repeated multiplication by weight_hh that makes gradients vanish or
# explode over long sequences - the reason LSTM and GRU exist.
for name,parms in rnn.named_parameters():
  if 'weight' in name:
    print(f'Parameter name : {name} | Parameter shape: {parms.shape}')

Parameter name : weight_ih_l0 | Parameter shape: torch.Size([16, 9])
Parameter name : weight_hh_l0 | Parameter shape: torch.Size([16, 16])


## Making RNN class

In [ ]:
# Wrapping nn.RNN in a Module with a Linear head - the standard pattern.
# The RNN produces a hidden state per step; the Linear turns each into a
# prediction.
# Return the hidden state too, so it can be inspected or carried forward.
class RNNnet(nn.Module):
  def __init__(self,input_size=9, hidden_size=16 , num_layers=1):
    super().__init__()

    self.input_size = input_size # Input length
    self.hidden_size = hidden_size # Size of RNN hidden layers
    self.num_layer = num_layers # Number of recurrent layers in rnns

    self.rnn = nn.RNN(input_size, hidden_size , num_layers)

    self.out = nn.Linear(hidden_size,1)

  def forward(self,x):

    y,hidden = self.rnn(x)

    return self.out(y),hidden

In [ ]:
test_rnn = RNNnet()
test_rnn

RNNnet(
  (rnn): RNN(9, 16)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

In [ ]:
# A forward pass with random data.
x = torch.randn(seq_length,batchsize,9)

y,h = test_rnn(x)

In [ ]:
# Output is (seq_len, batch, 1) - one prediction per time step.
# For a single prediction per sequence, take y[-1].
y.shape

torch.Size([5, 2, 1])

In [ ]:
# Hidden is (num_layers, batch, hidden_size). With stacked layers the states of
# all layers come back together, which is what 'stacked' means here.
h.shape # This is stacked

torch.Size([1, 2, 16])

# nn.RNN Parameters

`torch.nn.RNN(input_size, hidden_size, num_layers=1, nonlinearity='tanh', bias=True, batch_first=False, dropout=0.0, bidirectional=False)`

---

## 1. input_size (int)
Features per time step of input x.
Ex: word embeddings of dim 128 -> input_size=128

## 2. hidden_size (int)
Features in hidden state h -> memory capacity.
Params (single layer, no bias) = input_size * hidden_size + hidden_size^2.
With bias, add 2 * hidden_size.
For comparison: LSTM ~ 4 * (input_size + hidden_size) * hidden_size, GRU ~ 3x.

## 3. num_layers (int, default 1)
Stacked RNNs. Layer 2 sees layer 1's hidden states as input.
Stack -> hierarchical temporal patterns (char -> word -> phrase).

## 4. nonlinearity (str, default 'tanh')
Activation on the recurrent step:
- 'tanh' -> bounded [-1, 1], stable
- 'relu' -> unbounded, fights vanishing grads, can explode

## 5. bias (bool, default True)
Learnable biases $b_{ih}$, $b_{hh}$:

$$h_t = \tanh\left(W_{ih}\, x_t + b_{ih} + W_{hh}\, h_{t-1} + b_{hh}\right)$$

## 6. batch_first (bool, default False)
Tensor layout:
- False -> (seq_len, batch, features)
- True  -> (batch, seq_len, features)

## 7. dropout (float, default 0.0)
Dropout between layers (not recurrent, not last layer).
Active only if num_layers > 1. Try 0.3 for regularization.

## 8. bidirectional (bool, default False)
Run a 2nd RNN backwards -> concat/sum hidden states.
Doubles params and output dim. Use for full-sequence tasks (tagging), not streaming.

---

## I/O Shapes

| Tensor | Shape (default) | Shape (batch_first=True) |
|---|---|---|
| Input x | (seq_len, batch, input_size) | (batch, seq_len, input_size) |
| $h_0$ (optional) | (num_layers * num_directions, batch, hidden_size) | same |
| Output | (seq_len, batch, num_directions * hidden_size) | (batch, seq_len, num_directions * hidden_size) |
| $h_n$ | (num_layers * num_directions, batch, hidden_size) | same |

---

## Cheat Sheet

| Param | Controls |
|---|---|
| input_size | What RNN "sees" |
| hidden_size | What RNN "remembers" |
| num_layers | Depth of abstraction |
| nonlinearity | Squashing |
| bias | Affine offsets |
| batch_first | Tensor layout |
| dropout | Inter-layer reg |
| bidirectional | Past + future context |

---

## Parameter Count (single layer)

$$P_{\text{no bias}} = h \times (d + h)$$

$$P_{\text{with bias}} = P_{\text{no bias}} + 2h$$

where `d = input_size` and `h = hidden_size`.

Ex: `input_size=5, hidden_size=3` -> `3 * (5+3) = 24` (no bias), or `30` (with bias).